In [2]:
# functions to extract gates from a trained model
ALL_OPERATIONS = [
    "zero", "and", "not_implies", "a", "not_implied_by", "b", "xor", "or", 
    "not_or", "not_xor", "not_b", "implied_by", "not_a", "implies", "not_and", "one"
]

logic_layers = []
for i in model:
    if 'LogicLayer' in str(i):
        logic_layers.append(i)
        
def get_learned_gates_and_connections(model):
    """
    Extracts the learned logic gates and their connections from a trained DiffLogic model.

    Args:
        model: The trained DiffLogic model.

    Returns:
        learned_gates: A list of dictionaries containing the learned gate and input connections for each layer.
    """
    learned_gates = []

    for layer in logic_layers:
        if isinstance(layer, LogicLayer):
            layer_gates = []
            for neuron_idx in range(layer.weights.size(0)):
                # Get the learned gate by taking the argmax of the weights for the neuron
                gate_op_idx = layer.weights[neuron_idx].argmax().item()
                learned_gate = ALL_OPERATIONS[gate_op_idx]

                # Get the input connections (indices) for the gate
                input_neuron_a = layer.indices[0][neuron_idx].item()
                input_neuron_b = layer.indices[1][neuron_idx].item()

                layer_gates.append({
                    'Gate': learned_gate,
                    'Inputs': (input_neuron_a, input_neuron_b)
                })
            
            learned_gates.append(layer_gates)

    return learned_gates

# usage
#learned_gates = get_learned_gates_and_connections(model)

In [3]:
import torch
from difflogic import LogicLayer, GroupSum

# Prepare a small dataset for XOR (2-input XOR -> 2-class classification)
# Inputs as 2-bit vectors, Outputs as class 0 or 1 (we'll use class labels 0 and 1 for False/True)
X = torch.tensor([[0,0],[0,1],[1,0],[1,1]], dtype=torch.float32)  # all combinations of 2 bits
y = torch.tensor([0, 1, 1, 0], dtype=torch.long)  # XOR labels: 0⊕0=0, 0⊕1=1, 1⊕0=1, 1⊕1=0

# Define a simple DiffLogic model for binary classification
model = torch.nn.Sequential(
    LogicLayer(in_dim=2, out_dim=2, device='cuda', implementation='python', connections='random'),
    # out_dim=2 because we'll map to 2 output neurons (one per class) for GroupSum
    GroupSum(k=2, tau=1.0)  # Group 2 outputs into 2 classes; tau=1 for simplicity
)
model.train()  # ensure model in differentiable (training) mode

# Set up loss and optimizer
criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

# Training loop (small number of epochs since dataset is tiny)
for epoch in range(1000):
    # Forward pass
    logits = model(X)            # shape (4,2) for 4 samples and 2 class logits
    loss = criterion(logits, y)  
    print(loss)
    # Backpropagation
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

# Switch to evaluation mode to discretize the logic gates
model.eval()
learned_gates = get_learned_gates_and_connections(model)
print(learned_gates)

torch.int64 torch.int64
torch.int64 torch.int64


RuntimeError: Expected all tensors to be on the same device, but found at least two devices, cuda:0 and cpu!